In [12]:
from pathlib import Path
import pandas as pd
import nd2

images_dir = Path("/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/Sphere/220725_i11w-hT-M33-I76_sg1035_d10sphere")
images = list(sorted(images_dir.glob('*.nd2')))

image_index = 12

image_path = images[image_index]
print(image_path)
   
img_array = nd2.imread(image_path)
print(img_array.shape)
print(type(img_array))

tracks_df = pd.read_csv("/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/Sphere/220725_i11w-hT-M33-I76_sg1035_d10sphere/all_valid_20250605.csv")
tracks_df = tracks_df[tracks_df["image_name"] == image_path.name]

/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/Sphere/220725_i11w-hT-M33-I76_sg1035_d10sphere/sg100_Well7_1025.nd2
(33, 4, 2048, 2048)
<class 'numpy.ndarray'>


In [5]:
import napari

viewer = napari.Viewer()

viewer.add_image(img_array[:, 0], 
                 name='Halo-TDP-43')

viewer.add_image(img_array[:, 1], 
                 name='LYSOSOME')

viewer.add_image(img_array[:, 2], 
                 name='MITOCHONDRIA')

viewer.add_image(img_array[:, 3], 
                 name='BFP')

<Image layer 'BFP' at 0x31d801090>

In [45]:

import os
import numpy as np
import nd2
import imageio.v3 as iio
from PIL import Image, ImageDraw


# ---------------------------
# Normalization + rendering helpers
# ---------------------------

def normalize_crop_percentile(crop, lower=2, upper=98):
    """
    Robust per-channel percentile-based normalization.
    crop: (T, C, h, w) float/uint -> uint8 (T, C, h, w)
    Percentiles computed over ALL frames in the crop per channel.
    """
    T, C, h, w = crop.shape
    crop_norm = np.zeros((T, C, h, w), dtype=np.uint8)

    for c in range(C):
        channel_data = crop[:, c].reshape(-1).astype(np.float32)
        p_low, p_high = np.percentile(channel_data, [lower, upper])

        if (p_high - p_low) < 1e-3:
            crop_norm[:, c] = 0
        else:
            norm = (crop[:, c].astype(np.float32) - p_low) / (p_high - p_low)
            norm = np.clip(norm * 255.0, 0, 255)
            crop_norm[:, c] = norm.astype(np.uint8)

    return crop_norm

def _to_rgb(gray_u8):
    """(H,W) uint8 -> (H,W,3) uint8 grayscale RGB."""
    return np.stack([gray_u8, gray_u8, gray_u8], axis=-1)

def _apply_pseudocolor(gray_u8, rgb_color):
    """
    gray_u8: (H,W) uint8
    rgb_color: (R,G,B) in 0-255
    returns: (H,W,3) uint8
    """
    g = gray_u8.astype(np.float32) / 255.0
    color = np.array(rgb_color, dtype=np.float32) / 255.0
    out = g[..., None] * color[None, None, :]
    return (np.clip(out, 0, 1) * 255).astype(np.uint8)

def _merge_rgb(rgb_list):
    """Additive merge with clamp."""
    if len(rgb_list) == 0:
        raise ValueError("rgb_list is empty; nothing to merge.")
    acc = np.zeros_like(rgb_list[0], dtype=np.float32)
    for im in rgb_list:
        acc += im.astype(np.float32)
    return np.clip(acc, 0, 255).astype(np.uint8)

def _draw_overlays(rgb_u8, center_xy, track_xy,
                   circle_r=6, line_w=2,
                   circle_color=(255, 0, 0),      # red
                   line_color=(255, 255, 0)):     # yellow
    """
    Draw a circle at center and a polyline track.
    center_xy: (x,y) pixel coords in the current image
    track_xy: list[(x,y)] points (same coords)
    """
    im = Image.fromarray(rgb_u8)
    dr = ImageDraw.Draw(im)

    cx, cy = center_xy
    dr.ellipse((cx - circle_r, cy - circle_r, cx + circle_r, cy + circle_r),
               outline=circle_color, width=line_w)

    if len(track_xy) >= 2:
        dr.line(track_xy, fill=line_color, width=line_w, joint="curve")

    return np.array(im, dtype=np.uint8)

def _resize_nn(rgb_u8, scale):
    """Integer upscale via nearest-neighbor (keeps pixels crisp)."""
    if scale == 1:
        return rgb_u8
    im = Image.fromarray(rgb_u8)
    w, h = im.size
    im = im.resize((w * scale, h * scale), resample=Image.Resampling.NEAREST)
    return np.array(im, dtype=np.uint8)

def apply_gamma_u8(gray_u8, gamma=0.8):
    """
    Apply display gamma to uint8 grayscale image.
    gamma < 1  → brighten dim structures
    gamma > 1  → darken midtones
    """
    g = gray_u8.astype(np.float32) / 255.0
    g = np.power(g, gamma)
    return (np.clip(g, 0, 1) * 255).astype(np.uint8)



# ---------------------------
# Main writer
# ---------------------------

def save_track_gifs(
    img_array,               # (T, C, Y, X)
    track_df,                # must contain: particle, frame, y, x
    image_name,
    output_dir,
    padding=40,              # <-- bbox padding around entire track
    fps=10,
    upscale=2,               # 2 is usually a nice default
    norm_lower=2,
    norm_upper=98,
    channel_colors=None,     # dict: {0:(...), 1:(...), 2:(...)}
    composite_channels=(0, 1, 2),
    circle_r=6,
    line_w=2,
    gamma = 0.8,
    only_track_frames=True,  # write only frames where track exists
):
    output_dir = str(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    if img_array.ndim != 4:
        raise ValueError(f"Expected img_array shape (T,C,Y,X), got {img_array.shape}")

    T, C, Y, X = img_array.shape

    # Defaults requested
    if channel_colors is None:
        channel_colors = {
            0: (255, 0, 255),  # magenta
            1: (255, 0, 0),    # red
            2: (0, 255, 0),    # green
        }

    base_name = os.path.splitext(image_name)[0].replace(" ", "_")

    df = track_df.copy()
    df["frame"] = df["frame"].astype(int)
    df = df.sort_values(["particle", "frame"])

    duration = 1.0 / fps

    for pid, group in df.groupby("particle"):
        group = group.sort_values("frame")

        # Track-level bbox in full-image coords
        y_min, y_max = int(np.floor(group["y"].min())), int(np.ceil(group["y"].max()))
        x_min, x_max = int(np.floor(group["x"].min())), int(np.ceil(group["x"].max()))

        # Apply padding (your 40 px requirement)
        y1 = max(y_min - padding, 0)
        y2 = min(y_max + padding, Y)
        x1 = max(x_min - padding, 0)
        x2 = min(x_max + padding, X)

        if y2 <= y1 or x2 <= x1:
            continue

        # Crop all frames then normalize per-channel over all timepoints
        crop = img_array[:, :, y1:y2, x1:x2]  # (T, C, h, w)
        crop_norm = normalize_crop_percentile(crop, lower=norm_lower, upper=norm_upper)
        h, w = crop_norm.shape[2], crop_norm.shape[3]

        # Map frame -> center (in crop coords)
        centers = {
            int(r["frame"]): (float(r["x"]) - x1, float(r["y"]) - y1)
            for _, r in group.iterrows()
            if 0 <= int(r["frame"]) < T
        }

        # Progressive track line (grows over time)
        track_by_frame = {}
        running = []
        for t in sorted(centers.keys()):
            running.append(centers[t])
            track_by_frame[t] = running.copy()

        # Which frames to write
        if only_track_frames:
            t_list = np.array(sorted(centers.keys()), dtype=int)
        else:
            t_list = np.arange(T, dtype=int)

        if len(t_list) == 0:
            continue

        # Frame buffers
        per_ch_raw = [[] for _ in range(C)]
        per_ch_ann = [[] for _ in range(C)]
        comp_raw = []
        comp_ann = []

        def scale_pt(p):
            return (p[0] * upscale, p[1] * upscale)

        for t in t_list:
            center = centers.get(int(t), None)
            pts_now = track_by_frame.get(int(t), [])

            pseudo_list = []

            for c in range(C):
                gray = crop_norm[t, c]  
                # gray = apply_gamma_u8(gray, gamma=gamma)  

                # Per-channel grayscale
                rgb_raw = _to_rgb(gray)
                rgb_raw = _resize_nn(rgb_raw, upscale)

                rgb_ann = rgb_raw
                if center is not None:
                    rgb_ann = _draw_overlays(
                        rgb_raw.copy(),
                        center_xy=scale_pt(center),
                        track_xy=[scale_pt(p) for p in pts_now],
                        circle_r=circle_r * upscale,
                        line_w=max(1, line_w * upscale),
                        circle_color=(255, 0, 0),      # red circle
                        line_color=(255, 255, 0),      # yellow line
                    )

                per_ch_raw[c].append(rgb_raw)
                per_ch_ann[c].append(rgb_ann)

                # Composite uses only selected channels
                if c in composite_channels:
                    color = channel_colors.get(c, (255, 255, 255))
                    rgb_pc = _apply_pseudocolor(gray, color)
                    rgb_pc = _resize_nn(rgb_pc, upscale)
                    pseudo_list.append(rgb_pc)

            merged = _merge_rgb(pseudo_list) if len(pseudo_list) else np.zeros((h*upscale, w*upscale, 3), dtype=np.uint8)
            comp_raw.append(merged)

            merged_ann = merged
            if center is not None:
                merged_ann = _draw_overlays(
                    merged.copy(),
                    center_xy=scale_pt(center),
                    track_xy=[scale_pt(p) for p in pts_now],
                    circle_r=circle_r * upscale,
                    line_w=max(1, line_w * upscale),
                    circle_color=(255, 0, 0),      # red circle
                    line_color=(255, 255, 0),      # yellow line
                )
            comp_ann.append(merged_ann)

        # Write GIFs
        for c in range(C):
            fn_raw = os.path.join(output_dir, f"{base_name}_track{pid}_ch{c}_raw.gif")
            fn_ann = os.path.join(output_dir, f"{base_name}_track{pid}_ch{c}_annot.gif")
            iio.imwrite(fn_raw, per_ch_raw[c], duration=duration, loop=0)
            iio.imwrite(fn_ann, per_ch_ann[c], duration=duration, loop=0)

        fn_comp_raw = os.path.join(output_dir, f"{base_name}_track{pid}_composite_raw.gif")
        fn_comp_ann = os.path.join(output_dir, f"{base_name}_track{pid}_composite_annot.gif")
        iio.imwrite(fn_comp_raw, comp_raw, duration=duration, loop=0)
        iio.imwrite(fn_comp_ann, comp_ann, duration=duration, loop=0)

        print(f"[track {pid}] wrote gifs to: {output_dir}")


In [46]:
# Composite colors and channels (exclude ch4)
channel_colors = {
    0: (255, 0, 255),  # magenta
    1: (255, 0, 0),    # red
    2: (0, 255, 0),    # green
}
composite_channels = (0, 1, 2)
output_dir = "track_gifs"

# Write GIFs
save_track_gifs(
    img_array=img_array,
    track_df=tracks_df,
    image_name=image_path.name,
    output_dir=output_dir,
    padding=40,
    fps=10,
    upscale=2,
    norm_lower=2,
    norm_upper=98,
    channel_colors=channel_colors,
    composite_channels=composite_channels,
    circle_r=8,
    line_w=2,
    gamma = 1.2,
    only_track_frames=True,
)



[track 41] wrote gifs to: track_gifs
[track 55] wrote gifs to: track_gifs
[track 101] wrote gifs to: track_gifs
[track 110] wrote gifs to: track_gifs
[track 129] wrote gifs to: track_gifs
[track 406] wrote gifs to: track_gifs
[track 776] wrote gifs to: track_gifs
[track 1116] wrote gifs to: track_gifs
[track 1118] wrote gifs to: track_gifs
[track 1626] wrote gifs to: track_gifs
[track 1684] wrote gifs to: track_gifs


In [43]:
import numpy as np
import imageio.v3 as iio
from PIL import Image, ImageDraw

def make_all_tracks_gif(
    img_array,                # (T,C,Y,X)
    tracks_df,                # filtered to this image already
    out_path,
    fps=10,
    upscale=1,
    # display options:
    mode="composite",         # "composite" or "grayscale"
    grayscale_channel=0,      # if mode="grayscale"
    composite_channels=(0,1,2),
    channel_colors=None,      # dict for pseudocolor
    # intensity transform:
    norm_lower=2,
    norm_upper=98,
    gamma=0.8,
    # overlays:
    draw_track=True,
    draw_circle=True,
    circle_r=6,
    line_w=2,
    circle_color=(255, 0, 0),     # red
    line_color=(255, 255, 0),     # yellow
):
    assert img_array.ndim == 4, f"Expected (T,C,Y,X), got {img_array.shape}"
    T, C, Y, X = img_array.shape

    if channel_colors is None:
        channel_colors = {
            0: (255, 0, 255),  # magenta
            1: (255, 0, 0),    # red
            2: (0, 255, 0),    # green
        }

    # --- normalize full image per channel over the whole stack (stable movie look) ---
    # Compute percentiles per channel on the full stack (can be big but simplest).
    # If memory is an issue, we can do a running approximation.
    img_norm = np.zeros((T, C, Y, X), dtype=np.uint8)
    for c in range(C):
        data = img_array[:, c].reshape(-1).astype(np.float32)
        p_low, p_high = np.percentile(data, [norm_lower, norm_upper])
        if (p_high - p_low) < 1e-3:
            img_norm[:, c] = 0
        else:
            norm = (img_array[:, c].astype(np.float32) - p_low) / (p_high - p_low)
            img_norm[:, c] = np.clip(norm * 255.0, 0, 255).astype(np.uint8)

    def apply_gamma_u8(gray_u8, gamma=0.8):
        g = gray_u8.astype(np.float32) / 255.0
        g = np.power(g, gamma)
        return (np.clip(g, 0, 1) * 255).astype(np.uint8)

    def to_rgb(gray_u8):
        return np.stack([gray_u8, gray_u8, gray_u8], axis=-1)

    def apply_pseudocolor(gray_u8, rgb_color):
        g = gray_u8.astype(np.float32) / 255.0
        color = np.array(rgb_color, dtype=np.float32) / 255.0
        out = g[..., None] * color[None, None, :]
        return (np.clip(out, 0, 1) * 255).astype(np.uint8)

    def merge_add(rgb_list):
        acc = np.zeros_like(rgb_list[0], dtype=np.float32)
        for im in rgb_list:
            acc += im.astype(np.float32)
        return np.clip(acc, 0, 255).astype(np.uint8)

    def resize_nn(rgb_u8, scale):
        if scale == 1:
            return rgb_u8
        im = Image.fromarray(rgb_u8)
        w, h = im.size
        im = im.resize((w * scale, h * scale), resample=Image.Resampling.NEAREST)
        return np.array(im, dtype=np.uint8)

    # --- precompute per-particle points by frame (for fast overlay) ---
    df = tracks_df.copy()
    df["frame"] = df["frame"].astype(int)
    df = df.sort_values(["particle", "frame"])

    # positions_by_frame[t] = list of (x,y) centers at that time
    positions_by_frame = {t: [] for t in range(T)}
    # track_so_far_by_frame[t][pid] = list[(x,y)] up to time t
    track_so_far_by_frame = {t: {} for t in range(T)}

    for pid, g in df.groupby("particle"):
        pts = []
        for _, r in g.iterrows():
            t = int(r["frame"])
            if 0 <= t < T:
                pt = (float(r["x"]), float(r["y"]))
                pts.append((t, pt))

        # progressive build
        running = []
        for t, pt in pts:
            running.append(pt)
            positions_by_frame[t].append(pt)
            track_so_far_by_frame[t][pid] = running.copy()

    # Build frames
    frames = []
    for t in range(T):
        # background frame
        if mode == "grayscale":
            gray = img_norm[t, grayscale_channel]
            gray = apply_gamma_u8(gray, gamma=gamma)
            rgb = to_rgb(gray)
        elif mode == "composite":
            pseudo_list = []
            for c in composite_channels:
                if c < 0 or c >= C:
                    continue
                gray = img_norm[t, c]
                gray = apply_gamma_u8(gray, gamma=gamma)
                pseudo_list.append(apply_pseudocolor(gray, channel_colors.get(c, (255,255,255))))
            rgb = merge_add(pseudo_list) if pseudo_list else np.zeros((Y, X, 3), dtype=np.uint8)
        else:
            raise ValueError("mode must be 'grayscale' or 'composite'")

        rgb = resize_nn(rgb, upscale)

        # overlay tracks/circles
        if draw_track or draw_circle:
            im = Image.fromarray(rgb)
            dr = ImageDraw.Draw(im)

            # Draw track lines: for each particle that has points up to frame t
            if draw_track:
                # gather all polylines that exist at this frame
                # (only those with an entry at frame t — so the line grows when a new point exists)
                for pid, poly in track_so_far_by_frame[t].items():
                    if len(poly) >= 2:
                        poly_s = [(x * upscale, y * upscale) for (x, y) in poly]
                        dr.line(poly_s, fill=line_color, width=max(1, line_w * upscale), joint="curve")

            # Draw circles at positions at this frame
            if draw_circle and positions_by_frame[t]:
                r = circle_r * upscale
                w = max(1, line_w * upscale)
                for (x, y) in positions_by_frame[t]:
                    cx, cy = x * upscale, y * upscale
                    dr.ellipse((cx - r, cy - r, cx + r, cy + r), outline=circle_color, width=w)

            rgb = np.array(im, dtype=np.uint8)

        frames.append(rgb)

    iio.imwrite(str(out_path), frames, duration=1.0 / fps, loop=0)
    print("Saved:", out_path)

# Put output next to the ND2
whole_out = "track_gifs" / f"{image_path.stem}_ALLTRACKS_composite.gif"
whole_out.parent.mkdir(parents=True, exist_ok=True)

make_all_tracks_gif(
    img_array=img_array,
    tracks_df=tracks_df,
    out_path=whole_out,
    fps=10,
    upscale=1,                 # set 2 if you want bigger output
    mode="composite",          # or "grayscale"
    grayscale_channel=0,       # if mode="grayscale"
    composite_channels=(0,1,2),
    channel_colors={0:(255,0,255), 1:(255,0,0), 2:(0,255,0)},
    gamma=0.8,
    circle_color=(255,0,0),    # red
    line_color=(255,255,0),    # yellow
)



TypeError: unsupported operand type(s) for /: 'str' and 'str'